# S&P 500 Point-in-Time Constituents

Builds a point-in-time membership table for the S&P 500 index, to avoid
survivorship bias when backtesting `EquityBubbleRegimes`.

Source data: [`fja05680/sp500`](https://github.com/fja05680/sp500)
(MIT licensed), specifically
`S&P 500 Historical Components & Changes (Updated).csv` — one row per date
the index membership changed, listing the full set of tickers active on
that date.

**Coverage: 1996-01-02 to present.** Free sources (including this one) do
not cover 1990-1996; this is an accepted scope limitation (see
`context/project-overview.md`). For tickers present in the very first row
(1996-01-02), the true entry date is unknown — their `start_date` is
left-censored to 1996-01-02.

In [1]:
import os
import pandas as pd

In [2]:
# User Inputs
try:
    from google.colab import drive
    drive.mount('/content/drive')
    project_folder = '/content/drive/MyDrive/EquityBubbleRegimes'
    if not os.path.exists(project_folder):
        os.makedirs(project_folder)
except ImportError:
    project_folder = '.'

print(f'Project folder: {project_folder}')

Project folder: .


## 1. Fetch & Cache Raw Constituents Data

Downloads the historical components/changes CSV from `fja05680/sp500` and
caches a local copy (mirrors MacroRegimes2's `fred_raw.csv` caching pattern,
in case the upstream repo changes or disappears).

In [3]:
SOURCE_URL = (
    'https://raw.githubusercontent.com/fja05680/sp500/master/'
    'S%26P%20500%20Historical%20Components%20%26%20Changes%20(Updated).csv'
)
cache_path = f'{project_folder}/sp500_historical_components_raw.csv'

if os.path.exists(cache_path):
    print('Loading cached raw data...')
    raw = pd.read_csv(cache_path, parse_dates=['date'])
else:
    print('Fetching raw data from fja05680/sp500...')
    raw = pd.read_csv(SOURCE_URL, parse_dates=['date'])
    raw.to_csv(cache_path, index=False)

raw = raw.sort_values('date').reset_index(drop=True)
print(raw.shape)
print('Date range:', raw['date'].min().date(), 'to', raw['date'].max().date())
raw.head()

Loading cached raw data...
(2712, 2)
Date range: 1996-01-02 to 2026-06-02


,date,tickers
0,1996-01-02,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
1,1996-01-03,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
2,1996-01-04,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
3,1996-01-10,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
4,1996-01-11,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."


## 2. Transform to Point-in-Time Long Format

Each row in `raw` lists the *full set* of tickers active on that date. We
diff consecutive rows to find additions/removals, and build a long-format
table of `(ticker, start_date, end_date)` membership intervals.
`end_date` is `NaT` for tickers still active as of the latest date in the
source data.

In [4]:
raw['ticker_set'] = raw['tickers'].apply(lambda s: set(s.split(',')))

intervals = {}  # ticker -> list of [start_date, end_date]
prev_set = set()
prev_date = None

for _, row in raw.iterrows():
    date = row['date']
    cur_set = row['ticker_set']

    if prev_date is None:
        # First row: every ticker is left-censored to this date
        for ticker in cur_set:
            intervals.setdefault(ticker, []).append([date, None])
    else:
        added = cur_set - prev_set
        removed = prev_set - cur_set

        for ticker in added:
            intervals.setdefault(ticker, []).append([date, None])

        for ticker in removed:
            # Close out the most recently opened interval for this ticker
            if ticker in intervals and intervals[ticker][-1][1] is None:
                intervals[ticker][-1][1] = date

    prev_set = cur_set
    prev_date = date

rows = [
    (ticker, start, end)
    for ticker, ticker_intervals in intervals.items()
    for start, end in ticker_intervals
]

constituents = pd.DataFrame(rows, columns=['ticker', 'start_date', 'end_date'])
constituents = constituents.sort_values(['ticker', 'start_date']).reset_index(drop=True)
constituents.head()

,ticker,start_date,end_date
0,A,2000-06-05,NaT
1,AABA,1999-12-08,2017-06-19
2,AAL,1996-01-02,1997-01-15
3,AAL,2015-03-23,2024-09-23
4,AAMRQ,1996-01-02,2003-03-14


## 3. Validate

Spot-check well-known historical additions/removals, and sanity-check
summary stats (currently-active count should be ~500-505).

In [5]:
n_unique = constituents['ticker'].nunique()
n_active = constituents['end_date'].isna().sum()

print(f'Unique tickers ever in the index: {n_unique}')
print(f'Currently active tickers: {n_active}')
print(f'Date range: {constituents["start_date"].min().date()} to {raw["date"].max().date()}')

Unique tickers ever in the index: 1202
Currently active tickers: 503
Date range: 1996-01-02 to 2026-06-02


In [6]:
# Spot checks against known historical events
print('Tesla (added 2020-12-21):')
print(constituents[constituents['ticker'] == 'TSLA'], '\n')

print('Facebook -> Meta rename (FB added 2013-12-23, renamed to META 2022-06-09):')
print(constituents[constituents['ticker'].isin(['FB', 'META'])], '\n')

print('Enron (removed ~2001-11-30, ticker ENRNQ post-bankruptcy):')
print(constituents[constituents['ticker'] == 'ENRNQ'], '\n')

print('Google (added 2014-04-03 as GOOG, post Class C share split):')
print(constituents[constituents['ticker'] == 'GOOG'])

Tesla (added 2020-12-21):
     ticker start_date end_date
1122   TSLA 2020-12-21      NaT 

Facebook -> Meta rename (FB added 2013-12-23, renamed to META 2022-06-09):
    ticker start_date   end_date
434     FB 2013-12-23 2022-06-09
739   META 2022-06-09        NaT 

Enron (removed ~2001-11-30, ticker ENRNQ post-bankruptcy):
    ticker start_date   end_date
401  ENRNQ 1996-01-02 2001-11-30 

Google (added 2014-04-03 as GOOG, post Class C share split):
    ticker start_date end_date
520   GOOG 2014-04-03      NaT


## 4. Save Output

Long-format point-in-time membership table, one row per
`(ticker, start_date, end_date)` interval.

In [7]:
output_path = f'{project_folder}/EquityBubbleRegimes_SP500Constituents.csv'
constituents.to_csv(output_path, index=False)
print(f'Saved {len(constituents)} membership intervals to {output_path}')

Saved 1255 membership intervals to ./EquityBubbleRegimes_SP500Constituents.csv
